<a href="https://colab.research.google.com/github/chetanlohia0/Conversion-of-pdf-to-markdown/blob/main/Copy_of_Working_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python --version

import transformers

print("Transformers :", transformers.__version__)

Python 3.12.13
Transformers : 5.12.1


In [2]:
!pip uninstall -y transformers torch torchvision torchaudio

Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [1]:
!pip install \
torch==2.10.0 \
torchvision==0.25.0 \
transformers==4.57.1 \
accelerate \
sentencepiece \
Pillow==12.1.1 \
matplotlib==3.10.8 \
einops==0.8.2 \
addict==2.4.0 \
easydict==1.13 \
pymupdf==1.27.2.2 \
psutil==7.2.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 113.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 114.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [1]:
import torch
import transformers

print("=" * 60)
print("Python OK")
print("=" * 60)

print("Transformers :", transformers.__version__)
print("Torch        :", torch.__version__)

print()

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print("Memory :", round(props.total_memory / 1024**3, 2), "GB")

Python OK
Transformers : 4.57.1
Torch        : 2.10.0+cu128

CUDA Available : True
GPU : Tesla T4
Memory : 14.56 GB


In [2]:
import gc
import os
import traceback
from pathlib import Path

import fitz
import torch

from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "baidu/Unlimited-OCR"

MODEL = None
TOKENIZER = None

In [3]:
def gpu_status():

    if not torch.cuda.is_available():
        print("CUDA unavailable")
        return

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2

    print(f"Allocated : {allocated:.2f} MB")
    print(f"Reserved  : {reserved:.2f} MB")


def clear_gpu():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.synchronize()

        torch.cuda.empty_cache()

        torch.cuda.ipc_collect()

    gc.collect()

In [4]:
gpu_status()

Allocated : 0.00 MB
Reserved  : 0.00 MB


In [5]:
def load_model():

    global MODEL
    global TOKENIZER

    if MODEL is not None:

        print("Model already loaded.")

        return

    print("Loading model...")

    TOKENIZER = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )

    MODEL = AutoModel.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
        use_safetensors=True,
        torch_dtype=torch.bfloat16,
    )

    MODEL = MODEL.eval().cuda()

    print("Model loaded.")

    gpu_status()


def destroy_model():

    global MODEL
    global TOKENIZER

    print("Destroying model...")

    if MODEL is not None:

        del MODEL

    if TOKENIZER is not None:

        del TOKENIZER

    MODEL = None
    TOKENIZER = None

    clear_gpu()

    print("Model destroyed.")

    gpu_status()

In [6]:
gpu_status()

Allocated : 0.00 MB
Reserved  : 0.00 MB


In [7]:
def pdf_to_images(pdf_path, dpi=300):

    doc = fitz.open(pdf_path)

    temp_dir = Path(
        tempfile.mkdtemp(prefix="pdf_pages_")
    )

    pages = []

    matrix = fitz.Matrix(dpi / 72, dpi / 72)

    for page_number, page in enumerate(doc):

        img_path = temp_dir / f"page_{page_number+1:04d}.png"

        page.get_pixmap(matrix=matrix).save(str(img_path))

        pages.append(str(img_path))

    doc.close()

    return pages


In [8]:
from pathlib import Path

def batch_output_folder(batch):

    folder = OUTPUT_ROOT / (

        f"batch_{batch['id']}"

        f"_pages_{batch['start']:03d}_{batch['end']:03d}"

    )

    folder.mkdir(

        parents=True,

        exist_ok=True

    )

    return folder

In [9]:
from pathlib import Path

def validate_output(out_dir, expected_pages):

    md_file = out_dir / "result.md"

    if not md_file.exists():
        return False, "Missing result.md"

    text = md_file.read_text(
        encoding="utf8",
        errors="ignore"
    )

    # Empty file
    if len(text.strip()) < 100:
        return False, "Output too small"

    # Page count
    page_count = text.count("<PAGE>")

    if page_count != expected_pages:
        return False, (
            f"Expected {expected_pages} pages "
            f"but found {page_count}"
        )

    # OCR extracted almost nothing
    word_count = len(text.split())

    if word_count < expected_pages * 30:
        return False, (
            f"Too few words ({word_count})"
        )

    # Optional quality indicators
    has_table = "<table>" in text
    has_non_text = "[Non-Text]" in text

    if not has_table and not has_non_text:
        print("Warning: No tables or non-text regions detected.")

    return True, "OK"

In [10]:
SUCCESS = "SUCCESS"

OOM = "OOM"

ERROR = "ERROR"

INVALID = "INVALID"

SKIPPED = "SKIPPED"

In [11]:
from datetime import datetime

def log_batch(batch, status, message=""):

    out_dir = batch_output_folder(batch)

    log_file = out_dir / "batch.log"

    with open(log_file, "a", encoding="utf8") as f:

        f.write("=" * 60 + "\n")

        f.write(f"Time   : {datetime.now()}\n")

        f.write(f"Batch  : {batch['id']}\n")

        f.write(
            f"Pages  : {batch['start']} - {batch['end']}\n"
        )

        f.write(f"Status : {status}\n")

        if message:

            f.write(f"Message: {message}\n")

        f.write("\n")

In [12]:
def split_and_save_per_page(batch, out_dir):
    """Splits a multi-page batch Markdown file into individual page files."""
    md_file = out_dir / "result.md"
    if not md_file.exists():
        return
    text = md_file.read_text(encoding="utf8", errors="ignore")

    # Split tokens by <PAGE> indicator
    chunks = text.split("<PAGE>")
    if chunks and not chunks[0].strip():
        chunks = chunks[1:]

    pages_dir = OUTPUT_ROOT / "pages"
    pages_dir.mkdir(parents=True, exist_ok=True)

    start_page = batch["start"]
    for i, chunk in enumerate(chunks):
        current_page = start_page + i
        if current_page > batch["end"]:
            break
        page_file = pages_dir / f"page_{current_page:04d}.md"
        page_file.write_text(f"<PAGE>\n{chunk.strip()}", encoding="utf8")

In [13]:
import traceback

def process_batch(batch):

    out_dir = batch_output_folder(batch)

    done_file = out_dir / ".done"

    if done_file.exists():

        print(f"✓ Batch {batch['id']} already completed.")

        return SKIPPED

    print("\n" + "="*70)
    print(
        f"Batch {batch['id']} "
        f"({batch['start']} -> {batch['end']})"
    )
    print("="*70)

    try:
        import io
        import contextlib

        # Capture the raw terminal output stream to prevent the model from stripping layout tags
        f = io.StringIO()
        with contextlib.redirect_stdout(f):
            res = MODEL.infer_multi(
                TOKENIZER,
                prompt="<image><|grounding|>Multi page parsing.",
                image_files=batch["pages"],
                output_path=str(out_dir),
                image_size=1024,
                max_length=32768,
                no_repeat_ngram_size=35,
                ngram_window=1024,
                save_results=False, # Turned off so it doesn't dump clean markdown over our data
            )
        stdout_text = f.getvalue()

        # Keep printing to the screen so you can still track progress live
        print(stdout_text)

        # Select the true raw text string containing coordinates
        raw_text = res if (res and isinstance(res, str) and "<|det|>" in res) else stdout_text

        # Clean any extra terminal logs outside of the parsed document pages
        if "<PAGE>" in raw_text:
            raw_text = "<PAGE>" + raw_text.split("<PAGE>", 1)[1]
        if "===============save results:" in raw_text:
            raw_text = raw_text.split("===============save results:")[0]

        # Force save the RAW layout data with bounding boxes directly to result.md
        md_file = out_dir / "result.md"
        md_file.write_text(raw_text.strip(), encoding="utf8")

    except torch.OutOfMemoryError:

        clear_gpu()

        log_batch(
            batch,
            OOM,
            "CUDA Out Of Memory"
        )

        return OOM


    except RuntimeError as e:

        if "out of memory" in str(e).lower():

            clear_gpu()

            log_batch(
                batch,
                OOM,
                str(e)
            )

            return OOM

        traceback.print_exc()

        clear_gpu()

        log_batch(
            batch,
            ERROR,
            str(e)
        )

        return ERROR


    except Exception as e:

        traceback.print_exc()

        clear_gpu()

        log_batch(
            batch,
            ERROR,
            str(e)
        )

        return ERROR


    clear_gpu()

    # Validate based on the raw text file we just created
    valid, reason = validate_output(out_dir, len(batch["pages"]))
    if not valid:
        print(reason)
        log_batch(batch, INVALID, reason)
        return INVALID

    # --------------------------------------------------
    # New Saving Logic: Split raw text with bounding boxes
    # --------------------------------------------------
    md_file = out_dir / "result.md"
    if md_file.exists():
        text = md_file.read_text(encoding="utf8", errors="ignore")

        # Split by page token while preserving coordinate data tags
        chunks = text.split("<PAGE>")
        if chunks and not chunks[0].strip():
            chunks = chunks[1:]

        start_page = batch["start"]
        for i, chunk in enumerate(chunks):
            current_page = start_page + i
            if current_page > batch["end"]:
                break

            # Save to main output directory with bounding boxes completely preserved
            page_file = OUTPUT_ROOT / f"page_{current_page:04d}.md"
            page_file.write_text(f"<PAGE>\n{chunk.strip()}", encoding="utf8")
            print(f"Saved with BBoxes: {page_file.name}")

        # Remove temporary multi-page file
        md_file.unlink()

    log_batch(batch, SUCCESS)
    done_file.touch()
    print("✓ Batch completed.")
    return SUCCESS

In [14]:
def split_batch(batch):

    pages = batch["pages"]

    if len(pages) == 1:
        return []

    mid = len(pages) // 2

    left_pages = pages[:mid]
    right_pages = pages[mid:]

    left = {

        "id": batch["id"] + "A",

        "start": batch["start"],

        "end": batch["start"] + len(left_pages) - 1,

        "pages": left_pages

    }

    right = {

        "id": batch["id"] + "B",

        "start": left["end"] + 1,

        "end": batch["end"],

        "pages": right_pages

    }

    return [left, right]

In [15]:
FAILED_BATCHES = []
FINAL_FAILED_BATCHES = []  # New list to track true final failures

def process_failed_batches():
    global FAILED_BATCHES, FINAL_FAILED_BATCHES
    FINAL_FAILED_BATCHES = []

    while FAILED_BATCHES:
        batch = FAILED_BATCHES.pop(0)

        print("\n" + "="*70)
        print(f"Recovering Batch {batch['id']} (Pages: {batch['start']} -> {batch['end']})")
        print(f"Remaining Queue : {len(FAILED_BATCHES)}")
        print("="*70)

        children = split_batch(batch)

        # -------------------------------------------------------------------------
        # Handle Single-Page Failures (The Last Line of Defense)
        # -------------------------------------------------------------------------
        if len(children) == 0:
            current_page = batch["start"]
            print(f"⚠️ Single page {current_page} failed previously. Retrying ONE final time...")

            # Try processing the single page one last time
            status = process_batch(batch)
            clear_gpu()

            if status in [SUCCESS, SKIPPED]:
                print(f"✓ Single page {current_page} successfully recovered on final retry.")
                continue

            # If it STILL fails, rescue any partial text or write a placeholder file
            print(f"❌ Single page {current_page} completely failed. Rescuing data 'as-is'...")
            out_dir = batch_output_folder(batch)
            (out_dir / ".failed").touch()

            page_file = OUTPUT_ROOT / f"page_{current_page:04d}.md"
            md_file = out_dir / "result.md"
            rescued = False

            # Check if the model left any partial output in the batch folder before crashing
            if md_file.exists():
                try:
                    partial_text = md_file.read_text(encoding="utf8", errors="ignore").strip()
                    if partial_text:
                        if "<PAGE>" not in partial_text:
                            partial_text = f"<PAGE>\n{partial_text}"
                        page_file.write_text(partial_text, encoding="utf8")
                        print(f"💾 Saved partial raw output for page_{current_page:04d}.md")
                        rescued = True
                except Exception:
                    pass

            # If no partial file exists, write an explicit fallback token block so the sequence isn't broken
            if not rescued:
                fallback_text = f"<PAGE>\n[ERROR: Page parsing completely failed for page {current_page:04d}]\n"
                page_file.write_text(fallback_text, encoding="utf8")
                print(f"📝 Created fallback file for page_{current_page:04d}.md")

            # Append to our final tracking array so our counters show the correct failure metrics
            FINAL_FAILED_BATCHES.append(batch)
            continue

        for child in children:
            status = process_batch(child)
            clear_gpu()

            if status in [SUCCESS, SKIPPED]:
                continue

            FAILED_BATCHES.append(child)

In [16]:
def document_statistics():

    merged = OUTPUT_ROOT / "merged.md"

    text = merged.read_text(
        encoding="utf8",
        errors="ignore"
    )

    print("="*60)

    print("Document Statistics")

    print("="*60)

    print("Pages :", text.count("<PAGE>"))

    print("Tables :", text.count("<table>"))

    print("Non Text :", text.count("[Non-Text]"))

    print("Words :", len(text.split()))

    print("Characters :", len(text))

In [17]:
from google.colab import files
import tempfile
import fitz
import os
from pathlib import Path

while True:

    print("\n" + "="*80)
    print("UPLOAD PDF")
    print("="*80)

    uploaded = files.upload()

    if len(uploaded) == 0:
        print("No file uploaded.")
        break

    pdf_path = list(uploaded.keys())[0]

    pdf_name = Path(pdf_path).stem

    OUTPUT_ROOT = Path("outputs") / pdf_name
    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    print(f"\nPDF : {pdf_name}")

    # --------------------------------------------------
    # Convert PDF -> Images
    # --------------------------------------------------

    PAGES = pdf_to_images(pdf_path)

    print(f"Total Pages : {len(PAGES)}")

    # --------------------------------------------------
    # Create Batches
    # --------------------------------------------------


    BATCH_SIZE = 4

    BATCHES = []

    batch_no = 1

    for start in range(0, len(PAGES), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(PAGES))

        BATCHES.append({

            "id": f"B{batch_no:03d}",

            "start": start + 1,

            "end": end,

            "pages": PAGES[start:end]

        })

        batch_no += 1

    print(f"Total Batches : {len(BATCHES)}")


    # --------------------------------------------------
    # Load Model
    # --------------------------------------------------

    clear_gpu()

    load_model()

    # --------------------------------------------------
    # First Extraction Pass
    # --------------------------------------------------

    FAILED_BATCHES = []

    SUCCESS_BATCHES = []

    print("\n")
    print("="*80)
    print("FIRST PASS")
    print("="*80)

    for batch in BATCHES:

        status = process_batch(batch)

        if status in [SUCCESS, SKIPPED]:

            SUCCESS_BATCHES.append(batch)

        else:

            FAILED_BATCHES.append(batch)

    print()

    print("="*80)
    print("FIRST PASS SUMMARY")
    print("="*80)

    print("Successful :", len(SUCCESS_BATCHES))
    print("Failed     :", len(FAILED_BATCHES))

    # --------------------------------------------------
    # Recovery
    # --------------------------------------------------



UPLOAD PDF


Saving cclf.pdf to cclf.pdf

PDF : cclf
Total Pages : 70
Total Batches : 18
Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [744, 18, 850, 78]<|/det|>CMS
UNITED FOR MEDICAL & PRIVATE SERVICES
<|det|>header [858, 32, 943, 60]<|/det|>MEDICARE
SHARED SAVINGS
<|det|>header [859, 61, 908, 73]<|/det|>PROGRAM
<|det|>table [50, 111, 950, 681]<|/det|><table><tr><td>ELEMENT #</td><td>CLAIM FIELD LABEL</td><td>CLAIM FIELD NAME</td><td>START POSITION</td><td>END POSITION</td><td>DATA LENGTH</td><td>FORMAT</td><td>CLAIM FIELD DESCRIPTION</td></tr><tr><td>13</td><td>CLM_DEMO_3RD_NUM</td><td>Third Program Demonstration Number</td><td>57</td><td>58</td><td>2</td><td>X(2)</td><td>Medicare Demonstration Special Processing Number (SPN).This is a third demonstration number.This field will be used to hold the 2-byte number for future use with the Bundled Payments for Care Improvement initiative.</td></tr><tr><td>14</td><td>CLM_DEMO_4TH_NUM</td><td>Fourth Program Demonstration Number</td><td>59</td><td>60</td><td>2</td><td>X(2)</td><td>Medicare Demonstration Special Processing Number (SPN).This is a fourth de

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [743, 24, 842, 78]<|/det|>CMS
UNITED OFFICIAL MEDIA PARTNERSHIP
<|det|>header [858, 31, 944, 60]<|/det|>MEDICARE
SHARED SAVINGS
<|det|>header [858, 61, 907, 73]<|/det|>PROGRAM
<|det|>table [53, 111, 945, 852]<|/det|><table><tr><td>ELEMENT #</td><td>ELEMENT NAME</td><td>DATA DESCRIPTION</td><td>START POSITION</td><td>END POSITION</td><td>DATA LENGTH</td><td>FORMAT</td><td>COMMENTS</td></tr><tr><td>3</td><td>File Name</td><td>Name of CCLF file</td><td>9</td><td>51</td><td>43</td><td>X(43)</td><td>For file CCLF1, this field will contain “Part A Claims Header File”.For file CCLF2, this field will contain “Part A Claims Revenue Center Detail File”.For file CCLF3, this field will contain “Part A Procedure Code File”.For file CCLF4, this field will contain “Part A Diagnosis Code File”.For file CCLF5, this field will contain “Part B Physicians File”.For file CCLF6, this field will contain “Part B DME File”.For file CCLF7, this field will contain “Part D File”.For file CCLF8

No file uploaded.


In [18]:
if FAILED_BATCHES:
    print()
    print("=" * 80)
    print("RECOVERY PHASE")
    print("=" * 80)
    process_failed_batches()

print()
print("=" * 80)
print("RECOVERY COMPLETE")
print("=" * 80)

# Swapped out to read from the final failure tracker array
print("Remaining Failed :", len(FINAL_FAILED_BATCHES))
if FINAL_FAILED_BATCHES:
    failed_pages = [b["start"] for b in FINAL_FAILED_BATCHES]
    print(f"List of permanently failed pages: {failed_pages}")


RECOVERY PHASE

Recovering Batch B006 (Pages: 21 -> 24)
Remaining Queue : 5

Batch B006A (21 -> 22)


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [744, 32, 850, 81]<|/det|>CMS
UNITED OF AMERICAN VETERANS
<|det|>header [858, 32, 944, 46]<|/det|>MEDICARE
<|det|>header [858, 48, 944, 61]<|/det|>SHARED SAVINGS
<|det|>header [858, 62, 908, 75]<|/det|>PROGRAM
<|det|>table [50, 111, 951, 641]<|/det|><table><tr><td>ELEMENT #</td><td>CLAIM FIELD LABEL</td><td>CLAIM FIELD NAME</td><td>START POSITION</td><td>END POSITION</td><td>DATA LENGTH</td><td>FORMAT</td><td>CLAIM FIELD DESCRIPTION</td></tr><tr><td>4</td><td>CLM_TYPE_CD</td><td>Claim Type Code</td><td>36</td><td>37</td><td>2</td><td>9(02)</td><td>Signifies the type of claim being submitted through the Medicare or Medicaid programs.\( ^{H} \)Claim type codes are:10 = HHA claim20 = Non swing bed SNF claim30 = Swing bed SNF claim40 = Outpatient claim50 = Hospice claim60 = Inpatient claim61 = Inpatient “Full-Encounter” claim</td></tr><tr><td>5</td><td>CLM_VAL_SQNC_NUM</td><td>Claim Value Sequence Number</td><td>38</td><td>39</td><td>2</td><td>9(2)</td><td>An arbitrary 

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [744, 18, 850, 84]<|/det|>CMS
UNITED OF AMERICAN & INDIAN TERMINALS
<|det|>header [858, 32, 944, 75]<|/det|>MEDICARE
SHARED SAVINGS
PROGRAM
<|det|>text [50, 117, 600, 141]<|/det|>The filename convention for the Medicare Shared Savings Program in Table 4 is:
<|det|>text [51, 154, 504, 178]<|/det|>■ For regular CCLFs: P.A*****.ACO.ZC4Y**.Dymmdd.Thhmmst.
<|det|>text [51, 191, 748, 215]<|/det|>- For run-out CCLFs: P.A****.ACO.ZC4R**.Dymmdd.Thhmmst, "R" instead of "Y" indicating run-out.
<|det|>list [51, 154, 748, 215]<|/det|>
<|det|>text [51, 230, 410, 256]<|/det|>Table 4. Part A Diagnosis Code File (CCLF4)
<|det|>table [50, 264, 949, 613]<|/det|><table><tr><td>ELEMENT#</td><td>ELEMENT NAME</td><td>DATADESCRIPTION</td><td>START POSITION</td><td>END POSITION</td><td>DATALENGTH</td><td>FORMAT</td><td>COMMENTS</td></tr><tr><td>1</td><td>CUR_CLM_UNIQ_ID</td><td>Current ClaimUnique Identifier</td><td>1</td><td>13</td><td>13</td><td>9(13)</td><td>A unique identification numbe

...

<PAGE><|det|>header [744, 24, 842, 78]<|/det|>CMS
UNITED FOR MEDICAL & PRIVATE HEALTHCARE
<|det|>header [853, 32, 943, 60]<|/det|>MEDICARE
SHARED SAVINGS
<|det|>header [854, 61, 905, 73]<|/det|>PROGRAM
<|det|>text [50, 117, 600, 141]<|/det|>The filename convention for the Medicare Shared Savings Program in Table 9 is:
<|det|>text [51, 154, 503, 178]<|/det|>■ For regular CCLFs: P.A****.ACO.ZC9Y**.Dymmdd.Thhmmst.
<|det|>text [51, 191, 749, 215]<|/det|>- For run-out CCLFs: P.A****.ACO.ZC9R**.Dymmdd.Thhmmst, "R" instead of "Y" indicating run-out.
<|det|>list [51, 154, 749, 215]<|/det|>
<|det|>text [51, 230, 370, 255]<|/det|>Table 9. Beneficiary XREF File (CCLF9)
<|det|>table [50, 264, 945, 790]<|/det|><table><tr><td>ELEMENT#</td><td>BENEFICIARY FIELD LABEL</td><td>BENEFICIARY FIELD NAME</td><td>START POSITION</td><td>END POSITION</td><td>DATA LENGTH</td><td>FORMAT</td><td>BENEFICIARY FIELD DESCRIPTION</td></tr><tr><td>...</td><td>HICN_MBI_XREF_IND...</td><td>HICN/MBI XREF Indicator...</td>

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [744, 18, 850, 81]<|/det|>CMS
UNITED OFFICE OF MEDICAL & MEDICAL SERVICES
<|det|>header [858, 32, 945, 75]<|/det|>MEDICARE
SHARED SAVINGS
PROGRAM
<|det|>text [50, 116, 606, 141]<|/det|>The filename convention for the Medicare Shared Savings Program in Table 10 is:
<|det|>text [51, 154, 506, 180]<|/det|>■ For regular CCLFs: P.A****.ACO.ZCAY**.Dymmdd.Thhmmst.
<|det|>text [51, 192, 750, 217]<|/det|>- For run-out CCLFs: P.A****.ACO.ZCAR**.Dymmdd.Thhmmst, "R" instead of "Y" indicating run-out.
<|det|>list [51, 154, 750, 217]<|/det|>
<|det|>text [51, 230, 726, 257]<|/det|>Table 10. Part A Claims Benefit Enhancement and Demonstration Code File (CCLFA)
<|det|>table [52, 264, 949, 671]<|/det|><table><tr><td>ELEMENT No.</td><td>CLAIM FIELD LABEL</td><td>CLAIM FIELD NAME</td><td>START POSITION</td><td>END POSITION</td><td>DATA LENGTH</td><td>FORMAT</td><td>CLAIM FIELD DESCRIPTION</td></tr><tr><td>1</td><td>CUR_CLM_UNIQ_ID</td><td>Current Claim Unique Identifier</td><td>1</td><

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>header [744, 28, 850, 81]<|/det|>CMS
UNITED OFFICE OF THE SECURITY REGION
<|det|>header [858, 35, 943, 50]<|/det|>MEDICARE
<|det|>header [858, 51, 943, 63]<|/det|>SHARED SAVINGS
<|det|>header [858, 64, 911, 76]<|/det|>PROGRAM
<|det|>table [50, 111, 949, 778]<|/det|><table><tr><td>ELEMENT #</td><td>CLAIM FIELD LABEL</td><td>CLAIM FIELD NAME</td><td>START POSITION</td><td>END POSITION</td><td>DATA LENGTH</td><td>FORMAT</td><td>CLAIM FIELD DESCRIPTION</td></tr><tr><td>8</td><td>CLM_NGACO_SNF_VWR_SW</td><td>SNF 3-Day Waiver Benefit Enhancement Indicator</td><td>50</td><td>50</td><td>1</td><td>X(1)</td><td>A single character code of “Y” or “N” that indicates whether a particular claim is tied to a SNF 3-Day Waiver benefit enhancement. Blank if no data are available.Note: This field will be used for both NGACO and VT APM models.</td></tr><tr><td>9</td><td>CLM_NGACO_TLHLTH_SW</td><td>Telehealth Benefit Enhancement Indicator</td><td>51</td><td>51</td><td>1</td><td>X(1)</td><td>A s

In [ ]:
from pathlib import Path
import re

def merge_markdown():

    batch_dirs = list(OUTPUT_ROOT.glob("batch_*"))

    print(f"\nFound {len(batch_dirs)} batch folders")

    batches = []

    # -----------------------------------
    # Read metadata
    # -----------------------------------

    for folder in batch_dirs:

        log_file = folder / "batch.log"
        done_file = folder / ".done"
        md_file = folder / "result.md"

        if not done_file.exists():
            continue

        if not md_file.exists():
            continue

        m = re.search(
            r"batch_(B[0-9A-Z]+)_pages_(\d+)_(\d+)",
            folder.name
        )

        if m is None:
            continue

        batch_id = m.group(1)
        start = int(m.group(2))
        end = int(m.group(3))

        batches.append({

            "id": batch_id,
            "start": start,
            "end": end,
            "folder": folder

        })

    # -----------------------------------
    # Remove parent batches
    # -----------------------------------

    leaf_batches = []

    ids = [b["id"] for b in batches]

    for batch in batches:

        is_parent = False

        for other in ids:

            if other.startswith(batch["id"]) and other != batch["id"]:

                is_parent = True
                break

        if not is_parent:
            leaf_batches.append(batch)

    # -----------------------------------
    # Sort by page
    # -----------------------------------

    leaf_batches.sort(
        key=lambda x: x["start"]
    )

    print()

    print("Leaf batches used for merge:")

    for b in leaf_batches:

        print(
            f"{b['id']} : "
            f"{b['start']} -> {b['end']}"
        )

    # -----------------------------------
    # Merge
    # -----------------------------------

    merged = []

    for batch in leaf_batches:

        md = batch["folder"] / "result.md"

        with open(
            md,
            "r",
            encoding="utf8",
            errors="ignore"
        ) as f:

            merged.append(
                f.read().strip()
            )

    merged_file = OUTPUT_ROOT / "merged.md"

    with open(
        merged_file,
        "w",
        encoding="utf8"
    ) as f:

        f.write("\n\n".join(merged))

    print()

    print("Merged File Saved")

    print(merged_file)

    return merged_file

In [ ]:
merged_md = merge_markdown()


Found 22 batch folders

Leaf batches used for merge:
B001 : 1 -> 4
B002 : 5 -> 8
B003 : 9 -> 12
B004 : 13 -> 16
B005 : 17 -> 20
B006A : 21 -> 22
B006B : 23 -> 24
B007A : 25 -> 26
B007B : 27 -> 28
B008 : 29 -> 32
B009 : 33 -> 36
B010 : 37 -> 40
B011 : 41 -> 44
B012 : 45 -> 48
B013 : 49 -> 52
B014 : 53 -> 56
B015 : 57 -> 60
B016 : 61 -> 64
B017 : 65 -> 68
B018 : 69 -> 70

Merged File Saved
outputs/cclf/merged.md


In [ ]:
document_statistics()

Document Statistics
Pages : 70
Tables : 67
Non Text : 1
Words : 17737
Characters : 157412


In [23]:
import os
import shutil
from google.colab import files

# Define folder and zip names
folder_to_zip = 'outputs'
output_zip_file = 'outputs_archive'

# Check if the folder exists before zipping
if os.path.exists(folder_to_zip):
    print(f"Zipping the '{folder_to_zip}' folder... Please wait.")

    # Compress the folder into a .zip file
    shutil.make_archive(output_zip_file, 'zip', folder_to_zip)

    print("Zip file created successfully. Starting download...")

    # Download the file to your local computer


Zipping the 'outputs' folder... Please wait.
Zip file created successfully. Starting download...


In [19]:
import shutil

# Compress the 'outputs' folder into 'outputs.zip'
shutil.make_archive('/content/outputs', 'zip', '/content/outputs')


'/content/outputs.zip'